In [8]:
import requests
import time
from config import Config

def route_query(query, chat_history=None):
    url = f"    #url = Config.ROUTER_API
    headers = {
        "Authorization": f"Bearer {Config.API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": Config.REQUEST_ID_ROUTER,
        "Content-Type": "application/json"
    }
    data = {"query": query}
    if chat_history:
        data["chatHistory"] = chat_history

    while True:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 429:
            time.sleep(5)
            continue
        return response.json()

In [3]:
#라우터 성능 확인하기 위해 1차 확인!

query1= "선릉에 유명한 피잣집을 알려줘"
router_result1 = route_query(query1)

query2= "주식을 잘 하려면 무엇부터 공부해야할까?"
router_result2 = route_query(query2)

# JSON 전체 보기
print(router_result1)
print(router_result2)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '식당 검색', 'called': True}, 'blockedContent': {'result': [], 'called': False}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 530, 'completionTokens': 48, 'totalTokens': 578}}}
{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '주식', 'called': True}, 'blockedContent': {'result': [], 'called': True}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 1171, 'completionTokens': 80, 'totalTokens': 1251}}}


In [11]:
def get_chat_response(query, chat_history=None):
    # 먼저 라우터를 통해 도메인/차단 판단
    router_result = route_query(query, chat_history)

    # 목적 외 사용 판단 예시 (도메인이 '의료'면 차단)
    domain = router_result.get("result", {}).get("domain", {}).get("result", "")
    blocked_content = router_result.get("result", {}).get("blockedContent", {}).get("result", [])
    safety = router_result.get("result", {}).get("safety", {}).get("result", [])

    # 차단 도메인 목록 정의
    BLOCKED_DOMAINS = ["식당 검색", "주식"]

    if domain in BLOCKED_DOMAINS:
        return {
            "message" : "업무 외 질문이므로 해당 질문에 대해선 답변이 불가능합니다.",
            "filtered_domain" : domain
        }
    # 라우터에 걸리지 않았다면 정상적으로로 HCX 사용
    url = Config.CHAT_COMPLETIONS_API
    headers = {
        'Authorization': f'Bearer {Config.API_KEY}',
        'X-NCP-CLOVASTUDIO-REQUEST-ID': Config.REQUEST_ID_CHAT,
        'Content-Type': 'application/json',
    }

    system_prompt = "당신은 업무 도우미 입니다."
    messages = [{'role': 'system', 'content': system_prompt}]

    if chat_history:
        messages.extend(chat_history[-3:])
    else:
        messages.append({'role': 'user', 'content': query})

    data = {
        'messages': messages,
        "maxTokens": 512,
        "seed": 0,
        "temperature": 0.4,
        "topP": 0.4,
        "topK": 0,
        "repeatPenalty": 5.0
    }

    response = requests.post(url, headers=headers, json=data)
    return response.json()

In [12]:
response = get_chat_response("선릉에 삼겹살 집을 알려줘줘")

print(response)

{'message': '업무 외 질문이므로 해당 질문에 대해선 답변이 불가능합니다.', 'filtered_domain': '식당 검색'}


In [7]:
response = get_chat_response("삼성전자 주가는는 한달 전에 비해 어때?")

print(response)

{'message': '업무 외 질문이므로 해당 질문에 대해선 답변이 불가능합니다.', 'filtered_domain': '주식'}


In [5]:
response = get_chat_response("손흥민 선수는 22년도에 EPL에서 몇 골을 넣었어?")

print(response)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'message': {'role': 'assistant', 'content': '2022년 잉글랜드 프리미어리그(EPL)에서 손흥민 선수가 기록한 골은 총 23골입니다. 이는 아시아 선수 최초로 EPL 득점왕에 오른 기록이며, 손흥민 선수 개인적으로도 최고의 시즌이었습니다.\n\n토트넘 홋스퍼 FC 소속으로 활약하며 팀의 리그 4위와 UEFA 챔피언스리그 진출에도 큰 기여를 했습니다. \n\n다른 도움이 필요하시면 언제든지 말씀해 주세요.'}, 'inputLength': 21, 'outputLength': 87, 'stopReason': 'stop_before', 'seed': 1314922885}}
